# Incremental Gold Aggregation — Delta MERGE

**Objective:** Incrementally update daily financial KPIs using new Silver transactions, PySpark aggregations, and Delta MERGE.

### Why?

**For our project:** Incorporates new transactions into daily financial metrics while maintaining accurate transaction counts, status distributions, and completed amounts.

**For Data Engineering:** Demonstrates incremental change detection, selective aggregation, Delta MERGE, data reconciliation, and idempotency validation within the Medallion architecture.

#### 01 — Identify Incremental Silver Records

Read the Silver and Gold tables to verify their recorded state before incremental Gold processing.

Compare Silver Delta versions 0 and 1 using Time Travel and `exceptAll()` to identify the six transactions inserted during Batch 02 processing.

In [0]:
# Read the Silver table and verify the expected transaction count
df_silver_current = spark.table("workspace.default.fintech_transactions_silver")  
display(df_silver_current)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX007,C005,2026-10-18T13:20:00.000Z,PIX_IN,80.00,COMPLETED
TX002,C001,2026-10-18T09:30:00.000Z,PIX_OUT,70.00,COMPLETED
TX003,C002,2026-10-18T10:00:00.000Z,CARD_PURCHASE,90.50,COMPLETED
TX004,C003,2026-10-18T11:00:00.000Z,BILL_PAYMENT,130.00,COMPLETED
TX005,C004,2026-10-18T12:00:00.000Z,PIX_IN,400.00,PENDING
TX006,C003,2026-10-18T13:00:00.000Z,PIX_OUT,120.00,FAILED
TX008,C006,2026-10-18T14:00:00.000Z,PIX_OUT,35.00,COMPLETED
TX001,C001,2026-10-18T09:00:00.000Z,PIX_IN,250.00,COMPLETED
TX013,C003,2026-10-19T11:00:00.000Z,BILL_PAYMENT,60.00,COMPLETED
TX015,C005,2026-10-19T13:00:00.000Z,PIX_IN,900.00,FAILED


In [0]:
assert df_silver_current.count() == 14, "Unexpected silver table row count"

In [0]:
# Inspect the initial Gold aggregates before incremental processing
df_gold_existing = spark.table("workspace.default.fintech_transactions_daily_gold")  
display(df_gold_existing)

transaction_date,transaction_type,total_transactions,completed_transactions,pending_transactions,failed_transactions,completed_amount
2026-10-18,CARD_PURCHASE,1,1,0,0,90.50
2026-10-18,PIX_IN,3,2,1,0,330.00
2026-10-18,PIX_OUT,3,2,0,1,105.00
2026-10-18,BILL_PAYMENT,1,1,0,0,130.00


In [0]:
assert df_gold_existing.count() == 4, "Unexpected gold table row count"

In [0]:
# Identify newly inserted Silver records using Delta Time Travel
silver_table = "workspace.default.fintech_transactions_silver"

df_silver_v0 = spark.read.option("versionAsOf", 0).table(silver_table)
df_silver_v1 = spark.read.option("versionAsOf", 1).table(silver_table)

df_silver_increment = df_silver_v1.exceptAll(df_silver_v0)
display(df_silver_increment)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX013,C003,2026-10-19T11:00:00.000Z,BILL_PAYMENT,60.00,COMPLETED
TX015,C005,2026-10-19T13:00:00.000Z,PIX_IN,900.00,FAILED
TX017,C007,2026-10-19T15:00:00.000Z,PIX_IN,300.00,COMPLETED
TX014,C004,2026-10-19T12:00:00.000Z,PIX_OUT,30.00,PENDING
TX011,C001,2026-10-19T09:00:00.000Z,PIX_IN,150.00,COMPLETED
TX012,C002,2026-10-19T10:00:00.000Z,CARD_PURCHASE,45.25,COMPLETED


In [0]:
assert df_silver_v0.count() == 8
assert df_silver_v1.count() == 14
assert df_silver_increment.count() == 6


#### 02 — Identify Affected Gold Groups

Extract transaction dates from the incremental Silver records and identify distinct combinations of `transaction_date` and `transaction_type`.

These keys determine which Gold groups require recalculation, avoiding unnecessary processing of unaffected groups.

In [0]:
import pyspark.sql.functions as sf

# Derive the transaction date for incremental Gold aggregation
df_silver_increment = df_silver_increment.withColumn("transaction_date", sf.to_date(sf.col("transaction_ts")) )
display(df_silver_increment)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status,transaction_date
TX013,C003,2026-10-19T11:00:00.000Z,BILL_PAYMENT,60.00,COMPLETED,2026-10-19
TX015,C005,2026-10-19T13:00:00.000Z,PIX_IN,900.00,FAILED,2026-10-19
TX017,C007,2026-10-19T15:00:00.000Z,PIX_IN,300.00,COMPLETED,2026-10-19
TX014,C004,2026-10-19T12:00:00.000Z,PIX_OUT,30.00,PENDING,2026-10-19
TX011,C001,2026-10-19T09:00:00.000Z,PIX_IN,150.00,COMPLETED,2026-10-19
TX012,C002,2026-10-19T10:00:00.000Z,CARD_PURCHASE,45.25,COMPLETED,2026-10-19


In [0]:
# Identify distinct date and transaction type combinations requiring recalculation
df_gold_affected_keys = df_silver_increment.select("transaction_date", "transaction_type").distinct()
display(df_gold_affected_keys)

transaction_date,transaction_type
2026-10-19,BILL_PAYMENT
2026-10-19,PIX_IN
2026-10-19,PIX_OUT
2026-10-19,CARD_PURCHASE


In [0]:
assert df_gold_affected_keys.count() == 4

#### 03 — Recalculate and Validate Gold KPIs

Select all Silver transactions belonging to the affected date and transaction type groups. Recalculate total transactions, status counts, and completed amounts using the full set of records for each affected group.

Validate the aggregated results before updating Gold to ensure consistency and accurate financial reporting.

In [0]:
from pyspark.sql import functions as sf

# Prepare Silver transactions with the daily aggregation key
df_silver_prepared = df_silver_current.withColumn(
    "transaction_date",
    sf.to_date(sf.col("transaction_ts"))
)

display(df_silver_prepared)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status,transaction_date
TX007,C005,2026-10-18T13:20:00.000Z,PIX_IN,80.00,COMPLETED,2026-10-18
TX002,C001,2026-10-18T09:30:00.000Z,PIX_OUT,70.00,COMPLETED,2026-10-18
TX003,C002,2026-10-18T10:00:00.000Z,CARD_PURCHASE,90.50,COMPLETED,2026-10-18
TX004,C003,2026-10-18T11:00:00.000Z,BILL_PAYMENT,130.00,COMPLETED,2026-10-18
TX005,C004,2026-10-18T12:00:00.000Z,PIX_IN,400.00,PENDING,2026-10-18
TX006,C003,2026-10-18T13:00:00.000Z,PIX_OUT,120.00,FAILED,2026-10-18
TX008,C006,2026-10-18T14:00:00.000Z,PIX_OUT,35.00,COMPLETED,2026-10-18
TX001,C001,2026-10-18T09:00:00.000Z,PIX_IN,250.00,COMPLETED,2026-10-18
TX013,C003,2026-10-19T11:00:00.000Z,BILL_PAYMENT,60.00,COMPLETED,2026-10-19
TX015,C005,2026-10-19T13:00:00.000Z,PIX_IN,900.00,FAILED,2026-10-19


In [0]:
# Select all Silver records belonging to affected Gold groups
df_silver_affected = df_silver_prepared.join(
    df_gold_affected_keys,
    on=["transaction_date", "transaction_type"],
    how="inner"
)

display(df_silver_affected)

assert df_silver_affected.count() == 6

transaction_date,transaction_type,transaction_id,customer_id,transaction_ts,amount,status
2026-10-19,BILL_PAYMENT,TX013,C003,2026-10-19T11:00:00.000Z,60.00,COMPLETED
2026-10-19,CARD_PURCHASE,TX012,C002,2026-10-19T10:00:00.000Z,45.25,COMPLETED
2026-10-19,PIX_IN,TX011,C001,2026-10-19T09:00:00.000Z,150.00,COMPLETED
2026-10-19,PIX_OUT,TX014,C004,2026-10-19T12:00:00.000Z,30.00,PENDING
2026-10-19,PIX_IN,TX017,C007,2026-10-19T15:00:00.000Z,300.00,COMPLETED
2026-10-19,PIX_IN,TX015,C005,2026-10-19T13:00:00.000Z,900.00,FAILED


In [0]:
from pyspark.sql import functions as sf

# Recalculate daily KPIs for the affected date and transaction type groups
df_gold_incremental_source = df_silver_affected.groupBy(
    "transaction_date",
    "transaction_type"
).agg(
    sf.count("*").alias("total_transactions"),

    sf.sum(
        sf.when(sf.col("status") == "COMPLETED", 1).otherwise(0)
    ).alias("completed_transactions"),

    sf.sum(
        sf.when(sf.col("status") == "PENDING", 1).otherwise(0)
    ).alias("pending_transactions"),

    sf.sum(
        sf.when(sf.col("status") == "FAILED", 1).otherwise(0)
    ).alias("failed_transactions"),

    sf.sum(
        sf.when(
            sf.col("status") == "COMPLETED",
            sf.col("amount")
        ).otherwise(0)
    ).alias("completed_amount")
)

display(df_gold_incremental_source)

transaction_date,transaction_type,total_transactions,completed_transactions,pending_transactions,failed_transactions,completed_amount
2026-10-19,BILL_PAYMENT,1,1,0,0,60.00
2026-10-19,PIX_IN,3,2,0,1,450.00
2026-10-19,PIX_OUT,1,0,1,0,0.00
2026-10-19,CARD_PURCHASE,1,1,0,0,45.25


In [0]:
import pyspark.sql.functions as sf

# Verify that transaction status counts match the total in every affected group
status_total = (
    sf.col("completed_transactions")
    + sf.col("pending_transactions")
    + sf.col("failed_transactions")
)

df_inconsistent = df_gold_incremental_source.filter(
    status_total != sf.col("total_transactions")
)

assert df_inconsistent.count() == 0, "Unexpected inconsistent status in Gold Incremental source table"

In [0]:
assert df_gold_incremental_source.count() == 4

assert df_gold_incremental_source.agg(
    sf.sum("total_transactions")
).first()[0] == 6

assert df_gold_incremental_source.agg(
    sf.sum("completed_amount")
).first()[0] == 555.25

In [0]:
assert df_gold_incremental_source.agg(
    sf.sum("completed_transactions")
).first()[0] == 4

assert df_gold_incremental_source.agg(
    sf.sum("pending_transactions")
).first()[0] == 1

assert df_gold_incremental_source.agg(
    sf.sum("failed_transactions")
).first()[0] == 1

#### 04 — Merge Incremental Results into Gold

Apply Delta MERGE using `transaction_date` and `transaction_type` as the composite matching key.

Update existing Gold groups and insert new groups when no match exists. Validate the resulting table to confirm that each date and transaction type combination remains unique and that transaction status totals are consistent.

In [0]:
display(df_gold_incremental_source)

transaction_date,transaction_type,total_transactions,completed_transactions,pending_transactions,failed_transactions,completed_amount
2026-10-19,BILL_PAYMENT,1,1,0,0,60.00
2026-10-19,PIX_IN,3,2,0,1,450.00
2026-10-19,PIX_OUT,1,0,1,0,0.00
2026-10-19,CARD_PURCHASE,1,1,0,0,45.25


In [0]:
from delta.tables import DeltaTable

# Upsert recalculated Gold KPIs using the date and transaction type composite key
gold_table = DeltaTable.forName(
    spark,
    "workspace.default.fintech_transactions_daily_gold"
)

(
    gold_table.alias("target")
    .merge(
        df_gold_incremental_source.alias("source"),
        "target.transaction_date = source.transaction_date AND target.transaction_type = source.transaction_type"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
# Validate the updated Gold table and uniqueness of aggregation keys
df_gold_updated = spark.table(
    "workspace.default.fintech_transactions_daily_gold"
)

assert df_gold_updated.count() == 8,  "Unexpected daily gold table rows count"
assert df_gold_updated.select("transaction_date", "transaction_type").distinct().count() == 8, \
    "Unexpected gold groups count"

display(df_gold_updated)

transaction_date,transaction_type,total_transactions,completed_transactions,pending_transactions,failed_transactions,completed_amount
2026-10-18,CARD_PURCHASE,1,1,0,0,90.50
2026-10-18,PIX_IN,3,2,1,0,330.00
2026-10-18,PIX_OUT,3,2,0,1,105.00
2026-10-18,BILL_PAYMENT,1,1,0,0,130.00
2026-10-19,BILL_PAYMENT,1,1,0,0,60.00
2026-10-19,CARD_PURCHASE,1,1,0,0,45.25
2026-10-19,PIX_IN,3,2,0,1,450.00
2026-10-19,PIX_OUT,1,0,1,0,0.00


In [0]:
import pyspark.sql.functions as sf

# Verify status count consistency in the persisted Gold table
status_total = (
    sf.col("completed_transactions")
    + sf.col("pending_transactions")
    + sf.col("failed_transactions")
)

df_inconsistent = df_gold_updated.filter(
    status_total != sf.col("total_transactions")
)

assert df_inconsistent.count() == 0, "Unexpected inconsistent status in Gold updated table"

#### 05 — Reconcile Gold and Validate Idempotency

Reconcile the updated Gold KPIs with the Silver transactions, validating transaction totals, status counts, and completed financial amounts.

Repeat the same Delta MERGE and compare the resulting Gold records and Delta versions to verify that reprocessing preserves the aggregated business results without creating duplicate groups.

In [0]:
%sql
-- Inspect the initial Gold MERGE operation and its insertion metrics
DESCRIBE HISTORY workspace.default.fintech_transactions_daily_gold;

In [0]:
display(df_gold_updated)

transaction_date,transaction_type,total_transactions,completed_transactions,pending_transactions,failed_transactions,completed_amount
2026-10-18,CARD_PURCHASE,1,1,0,0,90.50
2026-10-18,PIX_IN,3,2,1,0,330.00
2026-10-18,PIX_OUT,3,2,0,1,105.00
2026-10-18,BILL_PAYMENT,1,1,0,0,130.00
2026-10-19,BILL_PAYMENT,1,1,0,0,60.00
2026-10-19,CARD_PURCHASE,1,1,0,0,45.25
2026-10-19,PIX_IN,3,2,0,1,450.00
2026-10-19,PIX_OUT,1,0,1,0,0.00


In [0]:
# Read Silver transactions for final reconciliation with Gold KPIs
df_silver = spark.table(
    "workspace.default.fintech_transactions_silver"
)

display(df_silver)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX007,C005,2026-10-18T13:20:00.000Z,PIX_IN,80.00,COMPLETED
TX002,C001,2026-10-18T09:30:00.000Z,PIX_OUT,70.00,COMPLETED
TX003,C002,2026-10-18T10:00:00.000Z,CARD_PURCHASE,90.50,COMPLETED
TX004,C003,2026-10-18T11:00:00.000Z,BILL_PAYMENT,130.00,COMPLETED
TX005,C004,2026-10-18T12:00:00.000Z,PIX_IN,400.00,PENDING
TX006,C003,2026-10-18T13:00:00.000Z,PIX_OUT,120.00,FAILED
TX008,C006,2026-10-18T14:00:00.000Z,PIX_OUT,35.00,COMPLETED
TX001,C001,2026-10-18T09:00:00.000Z,PIX_IN,250.00,COMPLETED
TX013,C003,2026-10-19T11:00:00.000Z,BILL_PAYMENT,60.00,COMPLETED
TX015,C005,2026-10-19T13:00:00.000Z,PIX_IN,900.00,FAILED


In [0]:
import pyspark.sql.functions as sf

# Reconcile Gold transaction counts, status totals, and completed amounts with Silver
assert (
    df_gold_updated.agg(sf.sum("total_transactions")).collect()[0][0]  == df_silver.count()
), "Unexpected rows count in the Gold and Silver tables"

assert (
    df_gold_updated.agg(sf.sum("completed_transactions")).collect()[0][0]  == df_silver.filter(sf.col("status") == "COMPLETED").count()
), "Unexpected rows count with CIMPLETED status in the Gold and Silver tables"

assert (
    df_gold_updated.agg(sf.sum("pending_transactions")).collect()[0][0]  == df_silver.filter(sf.col("status") == "PENDING").count()
), "Unexpected rows count with PENDING status in the Gold and Silver tables"

assert (
    df_gold_updated.agg(sf.sum("failed_transactions")).collect()[0][0]  == df_silver.filter(sf.col("status") == "FAILED").count()
), "Unexpected rows count with FAILED status in the Gold and Silver tables"

assert (
    df_gold_updated.agg(sf.sum("completed_amount")).collect()[0][0]  == df_silver.filter(sf.col("status") == "COMPLETED").agg(sf.sum("amount")).collect()[0][0]
), "Unexpected completed amount total in the Gold and Silver tables"

In [0]:
from delta.tables import DeltaTable

# Repeat the Gold MERGE to validate idempotent business results
gold_table = DeltaTable.forName(
    spark,
    "workspace.default.fintech_transactions_daily_gold"
)

(
    gold_table.alias("target")
    .merge(
        df_gold_incremental_source.alias("source"),
        "target.transaction_date = source.transaction_date AND target.transaction_type = source.transaction_type"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
DESCRIBE HISTORY workspace.default.fintech_transactions_daily_gold;

In [0]:
# Inspect the Gold table after the repeated MERGE
df_gold_idempotent = spark.table(
    "workspace.default.fintech_transactions_daily_gold"
)

display(df_gold_idempotent)

transaction_date,transaction_type,total_transactions,completed_transactions,pending_transactions,failed_transactions,completed_amount
2026-10-18,CARD_PURCHASE,1,1,0,0,90.50
2026-10-18,PIX_IN,3,2,1,0,330.00
2026-10-18,PIX_OUT,3,2,0,1,105.00
2026-10-18,BILL_PAYMENT,1,1,0,0,130.00
2026-10-19,BILL_PAYMENT,1,1,0,0,60.00
2026-10-19,CARD_PURCHASE,1,1,0,0,45.25
2026-10-19,PIX_IN,3,2,0,1,450.00
2026-10-19,PIX_OUT,1,0,1,0,0.00


In [0]:
import pyspark.sql.functions as sf

# Verify that all aggregated KPIs remain unchanged after reprocessing
assert (
    df_gold_updated.agg(sf.sum("total_transactions")).collect()[0][0]  
                        == df_gold_idempotent.agg(sf.sum("total_transactions")).collect()[0][0]
), "Unexpected rows count in the idenpotent gold table"

assert (
    df_gold_updated.agg(sf.sum("completed_transactions")).collect()[0][0]  
            == df_gold_idempotent.agg(sf.sum("completed_transactions")).collect()[0][0]  
), "Unexpected rows count with COMPLETED status in the idenpotent gold table"

assert (
    df_gold_updated.agg(sf.sum("pending_transactions")).collect()[0][0]  
                == df_gold_idempotent.agg(sf.sum("pending_transactions")).collect()[0][0]  
), "Unexpected rows count with PENDING status in the idenpotent gold table"

assert (
    df_gold_updated.agg(sf.sum("failed_transactions")).collect()[0][0]  
                        == df_gold_idempotent.agg(sf.sum("failed_transactions")).collect()[0][0]  
), "Unexpected rows count with FAILED status in the idenpotent gold table"

assert (
    df_gold_updated.agg(sf.sum("completed_amount")).collect()[0][0]  
                    == df_gold_idempotent.agg(sf.sum("completed_amount")).collect()[0][0]
), "Unexpected completed amount total in the idenpotent gold table"

In [0]:
# Compare Delta versions to confirm identical Gold row contents
gold_table = "workspace.default.fintech_transactions_daily_gold"

df_gold_v1 = spark.read.option("versionAsOf", 1).table(gold_table)
df_gold_v2 = spark.read.option("versionAsOf", 2).table(gold_table)

assert df_gold_v1.exceptAll(df_gold_v2).count() == 0
assert df_gold_v2.exceptAll(df_gold_v1).count() == 0

#### Engineering Note — Gold MERGE Idempotency

The incremental Gold MERGE uses `transaction_date` and `transaction_type` as composite keys to prevent duplicate aggregation groups.

Reprocessing the same data preserves the KPI values. However, `whenMatchedUpdateAll()` still updates matched rows and creates a new Delta table version, even when the resulting values are unchanged.

Therefore, this implementation provides logical idempotency but does not represent a physical no-op.